In [53]:
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
from adjustText import adjust_text


### **Puzzle 75 — Working with JSON**

players = [\
&emsp;{"name": "Aryn", "level": 12, "class": "Warrior", "stats": {"attack": 85, "defense": 72, "speed": 61}},\
&emsp;{"name": "Borin", "level": 8, "class": "Mage", "stats": {"attack": 55, "defense": 48, "speed": 79}},\
&emsp;{"name": "Cela", "level": 15, "class": "Rogue", "stats": {"attack": 91, "defense": 45, "speed": 95}},\
&emsp;{"name": "Dana", "level": 5, "class": "Paladin", "stats": {"attack": 67, "defense": 88, "speed": 52}}\
]

Write code that:

1. Converts the players list to a JSON string using json.dumps() and prints it
2. Prints the JSON string with pretty formatting — look up the indent parameter
3. Saves the players data to a file called players.json using json.dump()
4. Reads it back from the file using json.load() and stores it in a new variable
5. Loops through the loaded data and prints each player's name and their highest stat

In [ ]:
players = [
    {"name": "Aryn", "level": 12, "class": "Warrior", "stats": {"attack": 85, "defense": 72, "speed": 61}},
    {"name": "Borin", "level": 8, "class": "Mage", "stats": {"attack": 55, "defense": 48, "speed": 79}},
    {"name": "Cela", "level": 15, "class": "Rogue", "stats": {"attack": 91, "defense": 45, "speed": 95}},
    {"name": "Dana", "level": 5, "class": "Paladin", "stats": {"attack": 67, "defense": 88, "speed": 52}}
]

example = json.dumps(players)
print(example)
print(type(example))

format_example = json.dumps(players, indent = 2)
print(format_example)

with open('data/players.json', 'w') as f:
  json.dump(players, f)

with open('data/players.json', 'r') as f:
  loaded = json.load(f)
  for line in loaded:
    max_stat = max(line["stats"], key=lambda s: line["stats"][s])
    print(f"{line['name']}'s highest stat is: {max_stat}")

### **Puzzle 76 — GET Requests**

Using https://jsonplaceholder.typicode.com (a free fake REST API built for practice), write code that:

1. Makes a GET request to https://jsonplaceholder.typicode.com/users and prints the status code
2. Parses the response as JSON and prints how many users there are
3. Loops through the users and prints each user's name and email
4. Makes a GET request for a single user at https://jsonplaceholder.typicode.com/users/1 and prints their name, email, and city (city is nested inside an address object)
5. Handles the case where the request fails by checking the status code before trying to parse

In [ ]:
url = 'https://jsonplaceholder.typicode.com/users'
r = requests.get(url)
print(r.status_code)

if r.status_code == 200:
  data = r.json()
  print(len(data))
else:
  print('An error has occured')
for user in data:
  print(f'{user['name']}: {user['email']}')

url2 = 'https://jsonplaceholder.typicode.com/users/1'
r2 = requests.get(url2)
if r2.status_code == 200:
  data2 = r2.json()
  print(f'Name: {data2['name']}, Email: {data2['email']}, City: {data2['address']['city']}')
else:
  print('An error has occured')

### **Puzzle 77 — POST Requests and Query Parameters**
Two new concepts this time:\
Query parameters — extra filters added to a URL:

Instead of https://api.com/posts?userId=1 \
params = {"userId": 1}\
response = requests.get(url, params=params)

POST requests — sending data to create something:\
payload = {"title": "My Post", "body": "Hello world", "userId": 1}\
response = requests.post(url, json=payload)

Write code that:

1. Makes a GET request to https://jsonplaceholder.typicode.com/posts with a query parameter userId=1 to get only posts by user 1 — print how many posts they have
2. Prints the title of each of their posts
3. Makes a POST request to https://jsonplaceholder.typicode.com/posts creating a new post with any title and body you want, with userId=1
4. Prints the status code of the POST response and the created post's data
5. Makes a GET request to https://jsonplaceholder.typicode.com/posts/1/comments and prints the name and email of each commenter

In [ ]:
r = requests.get('https://jsonplaceholder.typicode.com/posts')
r2 = requests.get('https://jsonplaceholder.typicode.com/posts/1/comments')
print(json.dumps(r.json()[0], indent=2))  # just look at the first item
#REMINDER: Always look at the first item in a list of JSON data to understand the structure of the data you are working with.
    #Can also use pd.DataFrame(r.json()) to see the data in a table format, but this is not always the best way to view the data.
print(json.dumps(r2.json()[0], indent=2))  # just look at the first item

In [ ]:
url = 'https://jsonplaceholder.typicode.com/posts'
url2 = 'https://jsonplaceholder.typicode.com/posts/1/comments'
params = {'userId': 1}
r = requests.get(url, params=params)
print(f"Number of posts by user 1: {len(r.json())}")
for post in r.json():
    print(post['title'])
p = requests.post(url, json={'userId': 1, 'title': 'New Post', 'body': 'This is the body of the new post.'})
print(f"Status code: {p.status_code}")
print(json.dumps(p.json(), indent=2))
r2 = requests.get(url2)
for comment in r2.json():
    print(f"Name: {comment['name']}, Email: {comment['email']}")

### **Puzzle 78 — Combining APIs with pandas**
This is where it gets really practical — pulling API data directly into a DataFrame for analysis.\

Using https://jsonplaceholder.typicode.com:

1. Fetch all posts from /posts and load into a DataFrame
2. Fetch all users from /users and load into a DataFrame
3. Merge them so each post shows the author's name and email (hint: both have a userId/id field to join on — think about what you'd need to rename)
4. Find which user has written the most posts
5. Print the top 3 most prolific posters with their post count

In [ ]:
url = 'https://jsonplaceholder.typicode.com/posts'
url2 = 'https://jsonplaceholder.typicode.com/users'
r = requests.get(url)
r2 = requests.get(url2)
posts_df = pd.DataFrame(r.json())
users_df = pd.DataFrame(r2.json())
data_df = pd.merge(posts_df, users_df, left_on='userId', right_on='id', how='left')

print(data_df[['title', 'name', 'email']])
post_counts = data_df.groupby('name').agg(post_count=('id_x', 'count')).sort_values('post_count', ascending=False)
print(post_counts)
max_count = post_counts['post_count'].max()
top_posters = post_counts[post_counts['post_count'] == max_count]
if len(top_posters) > 1:
    print(f"It's a {len(top_posters)}-way tie at {max_count} posts each:")
    for name in top_posters.index:
        print(f"  - {name}")
else:
    print(f"User with the most posts: {top_posters.index[0]} with {max_count} posts.")

print(f'The top 3 users with the most posts are:\n{post_counts.head(3)}')




### **Puzzle 79 — API + Pandas Analysis**

Using https://jsonplaceholder.typicode.com:

1. Fetch all comments from /comments and load into a DataFrame — explore the raw response first with json.dumps() so you know the field names
2. Fetch all posts from /posts and load into a DataFrame
3. Merge them so each comment shows its post's title
4. Find which post has the most comments
5. Find the top 5 most active commenters by email address
6. Create a new column "email_domain" that extracts just the domain from each email address (everything after the @ sign)
7. Find which email domain appears most frequently across all comments

In [ ]:
url = 'https://jsonplaceholder.typicode.com/comments'
r = requests.get(url)
print(json.dumps(r.json()[0], indent=2))
df = pd.DataFrame(r.json())

url2 = 'https://jsonplaceholder.typicode.com/posts'
r2 = requests.get(url2)
print(json.dumps(r2.json()[0], indent=2))
df2 = pd.DataFrame(r2.json())
data_df = pd.merge(df, df2, left_on= 'postId', right_on='id', suffixes=('_comment', '_post'))
print(data_df.head())
post_counts = data_df.groupby('postId').agg(post_count=('id_comment', 'count')).sort_values('post_count', ascending=False)
print(post_counts)
max_counts = post_counts['post_count'].max()
most_comments = post_counts[post_counts['post_count'] == max_counts]
print(most_comments)
if len(most_comments) > 1:
    print(f"It's a {len(most_comments)}-way tie with {max_counts} comments each!")
    for post_id in most_comments.index:
        title = data_df[data_df['postId'] == post_id]['title'].iloc[0]
        print(f' - Post {post_id}: {title}')
else:
    post_id = most_comments.index[0]  
    title = data_df[data_df['postId'] == post_id]['title'].iloc[0]  
    print(f'Post with the most comments: Post {post_id}: {title} with {max_counts} comments')

top_commenters = data_df.groupby('email').agg(comment_count=('id_comment', 'count')).sort_values('comment_count', ascending=False)
print(top_commenters.head(5))
data_df['email_domain'] = data_df['email'].str.split('@').str[1].str.lower()
freq_domain = data_df.groupby('email_domain').agg(frequency=('id_comment', 'count')).sort_values('frequency', ascending=False)
print(freq_domain)

### **Puzzle 80 — PokeAPI Analysis**

1. Fetch the first 50 Pokémon from https://pokeapi.co/api/v2/pokemon?limit=50 — explore the raw response first
2. You'll notice each Pokémon just has a name and url — fetch the details for each one by making individual requests to their URLs and extract name, base_experience, height, and weight
3. Load into a DataFrame
4. Find the tallest, heaviest, and highest base experience Pokémon
5. Plot a scatter plot of height vs weight with Pokémon names as hover labels

In [ ]:
url= 'https://pokeapi.co/api/v2/pokemon?limit=50'
r = requests.get(url)
print(r.status_code)
print(json.dumps(r.json(), indent = 2))
poke_data = []
for mon in r.json()['results']:
    details = requests.get(mon['url']).json()
    poke_data.append({
        'name': details['name'],
        'base_experience': details['base_experience'],
        'height': details['height'],
        'weight': details['weight']
    })
poke_df = pd.DataFrame(poke_data)
print(poke_df)
max_height = poke_df['height'].max()
tallest = poke_df[poke_df['height'] == max_height]
if len(tallest) > 1:
    print(f'There is a {len(tallest)}-way tie at height {max_height}!')
    for idx in tallest.index:
       print(f'Tallest Pokemon - {tallest.loc[idx, 'name']}: {tallest.loc[idx, 'height']}')
else:
    print(f'Tallest Pokemon - {tallest['name'].iloc[0]}: {tallest['height'].iloc[0]}')

max_weight = poke_df['weight'].max()
heaviest = poke_df[poke_df['weight'] == max_weight]
if len(heaviest) > 1:
    print(f'There is a {len(heaviest)}-way tie at weight {max_weight}!')
    for idx in heaviest.index:
       print(f'Heaviest Pokemon - {heaviest.loc[idx, 'name']}: {heaviest.loc[idx, 'weight']}')
else:
    print(f'Heaviest Pokemon - {heaviest['name'].iloc[0]}: {heaviest['weight'].iloc[0]}')

max_exp = poke_df['base_experience'].max()
strongest = poke_df[poke_df['base_experience'] == max_exp]
if len(strongest) > 1:
    print(f'There is a {len(strongest)}-way tie at base experience {max_exp}!')
    for idx in strongest.index:
       print(f'Strongest Pokemon - {strongest.loc[idx, 'name']}: {strongest.loc[idx, 'base_experience']}')
else:
    print(f'Strongest Pokemon - {strongest['name'].iloc[0]}: {strongest['base_experience'].iloc[0]}')

plt.figure(figsize=(20, 20))
plt.scatter(x=poke_df['height'], y=poke_df['weight'], marker='^', color='red')
texts = []
for i, label in enumerate(poke_df['name']):
    texts.append(plt.text(
        poke_df['height'].iloc[i],
        poke_df['weight'].iloc[i],
        label,
        fontsize=9
    ))
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))
plt.show()
    
    